# recursion
**Prerequisites:** function_basics, parameters_and_arguments, return_values, scope

**Outcome:** After this notebook you will know exactly how recursion works at the call stack level, how to identify the base case and recursive case in any problem, the cost of recursion in Python, and when to use iteration instead.

## The Problem

In [ ]:
# a nested config structure — how do you flatten it without knowing the depth?
config = {
    "database": {
        "host": "localhost",
        "port": 5432,
        "credentials": {
            "user": "admin",
            "password": "secret"
        }
    },
    "region": "us-east",
    "retries": 3
}

# with loops you need to know the depth in advance
# the structure could be 1 level deep or 10 levels deep
# a loop cannot handle unknown depth cleanly — recursion can

## The Concept

Recursion is when a function calls itself to solve a smaller version of the same problem. Every recursive solution has two parts: a base case — the simplest input where the answer is known directly and no further call is needed — and a recursive case — where the function calls itself with a smaller or simpler input and builds the answer from the result. Each call creates a new stack frame with its own local variables. The frames accumulate until the base case is reached, then the results unwind back through the chain. Python enforces a call stack limit — by default around 1000 frames.

## Minimal Example

In [ ]:
def countdown(n):
    if n <= 0:              # base case — stop here
        print("done")
        return
    print(n)
    countdown(n - 1)        # recursive case — smaller input

countdown(5)

## How Python Does It

Each recursive call pushes a new frame onto the call stack. The frame holds the local variables for that call — including `n` in the countdown example. When the base case is reached the innermost frame returns, Python pops it, and execution continues in the frame below. This repeats until all frames have returned. If the base case is never reached — or the input is too large — Python runs out of stack space and raises `RecursionError`. You can inspect and change the limit with `sys.getrecursionlimit()` and `sys.setrecursionlimit()`.

In [ ]:
import sys

print(sys.getrecursionlimit())   # 1000 by default

# trace the call stack depth during recursion
def depth_demo(n):
    frame_depth = len([f for f in iter(lambda: sys._getframe(0), None)
                       if False]) or __import__('traceback').extract_stack().__len__()
    print(f"n={n}  stack depth={frame_depth}")
    if n == 0:
        return
    depth_demo(n - 1)

depth_demo(4)

## Building Up

In [ ]:
# factorial — the textbook example
def factorial(n):
    if n == 0:                      # base case
        return 1
    return n * factorial(n - 1)     # recursive case

print(factorial(0))   # 1
print(factorial(5))   # 120
print(factorial(10))  # 3628800

In [ ]:
# sum of a list — recursive version
def recursive_sum(values):
    if not values:                              # base case — empty list
        return 0
    return values[0] + recursive_sum(values[1:])  # head + sum of tail

print(recursive_sum([1, 2, 3, 4, 5]))   # 15
print(recursive_sum([]))                # 0

In [ ]:
# flatten a nested list of unknown depth
def flatten(data):
    result = []
    for item in data:
        if isinstance(item, list):      # recursive case — item is a list
            result.extend(flatten(item))
        else:                           # base case — item is a value
            result.append(item)
    return result

nested = [1, [2, 3], [4, [5, [6, 7]]], 8]
print(flatten(nested))   # [1, 2, 3, 4, 5, 6, 7, 8]

In [ ]:
# solving the original problem — flatten a nested dict
def flatten_dict(d, prefix=""):
    result = {}
    for key, value in d.items():
        full_key = f"{prefix}.{key}" if prefix else key
        if isinstance(value, dict):          # recursive case
            result.update(flatten_dict(value, full_key))
        else:                                # base case
            result[full_key] = value
    return result

config = {
    "database": {
        "host": "localhost",
        "port": 5432,
        "credentials": {"user": "admin", "password": "secret"}
    },
    "region": "us-east",
    "retries": 3
}

flat = flatten_dict(config)
for k, v in flat.items():
    print(f"{k}: {v}")

In [ ]:
# memoisation — cache results to avoid recomputing
def fibonacci(n, memo={}):
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fibonacci(n - 1, memo) + fibonacci(n - 2, memo)
    return memo[n]

print(fibonacci(10))   # 55
print(fibonacci(40))   # 102334155 — fast because of cache

In [ ]:
# using functools.lru_cache for cleaner memoisation
from functools import lru_cache

@lru_cache(maxsize=None)
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(50))    # 12586269025
print(fibonacci.cache_info())

## Where It Breaks

In [ ]:
# missing base case — infinite recursion — RecursionError
def broken_countdown(n):
    print(n)
    broken_countdown(n - 1)   # no base case — never stops

try:
    broken_countdown(5)
except RecursionError as e:
    print(f"RecursionError: {e}")

In [ ]:
# deep recursion on large input hits Python's stack limit
import sys

def recursive_sum(values):
    if not values:
        return 0
    return values[0] + recursive_sum(values[1:])

try:
    result = recursive_sum(list(range(2000)))   # exceeds default limit of ~1000
except RecursionError as e:
    print(f"RecursionError on large input: {e}")

# the iterative version handles any size
print(sum(range(2000)))   # 1999000

In [ ]:
# slicing creates a new list on every call — O(n²) time, O(n²) memory
def slow_sum(values):
    if not values:
        return 0
    return values[0] + slow_sum(values[1:])   # values[1:] copies the rest each time

# for 1000 elements this creates 1000 list copies
# fix: pass an index instead of slicing

## The Fix

In [ ]:
# fix slicing — pass index, no copies created
def recursive_sum(values, index=0):
    if index == len(values):   # base case
        return 0
    return values[index] + recursive_sum(values, index + 1)

print(recursive_sum([1, 2, 3, 4, 5]))   # 15

In [ ]:
# when recursion depth is a concern — convert to iteration
# iterative flatten using an explicit stack
def flatten_iterative(data):
    result = []
    stack  = list(data)        # initialise stack with top-level items
    while stack:
        item = stack.pop(0)
        if isinstance(item, list):
            stack = item + stack   # push inner items to front
        else:
            result.append(item)
    return result

nested = [1, [2, 3], [4, [5, [6, 7]]], 8]
print(flatten_iterative(nested))   # [1, 2, 3, 4, 5, 6, 7, 8]

## In a Real System

In [ ]:
# traversing a tree of pipeline stages
# each stage may have sub-stages — depth is unknown at design time

pipeline_tree = {
    "name": "etl",
    "status": "done",
    "children": [
        {
            "name": "extract",
            "status": "done",
            "children": [
                {"name": "read_db",    "status": "done",   "children": []},
                {"name": "read_files", "status": "failed", "children": []},
            ]
        },
        {
            "name": "transform",
            "status": "done",
            "children": [
                {"name": "validate",  "status": "done", "children": []},
                {"name": "normalise", "status": "done", "children": []},
            ]
        },
    ]
}

def collect_failures(stage):
    failures = []
    if stage["status"] == "failed":           # base case: this stage failed
        failures.append(stage["name"])
    for child in stage["children"]:           # recursive case: check children
        failures.extend(collect_failures(child))
    return failures

print(collect_failures(pipeline_tree))   # ["read_files"]

## Performance

Python does not optimise tail recursion — every call adds a frame to the stack regardless of position. The default stack limit is around 1000 frames. For problems with input size that could exceed a few hundred, use iteration or increase the limit with `sys.setrecursionlimit()`. Memoisation converts exponential recursive algorithms to linear time but does not reduce stack depth. For deep trees and graphs, convert to iterative using an explicit stack data structure — this gives you unlimited depth and predictable memory usage.

## Anti-Pattern

In [ ]:
# anti-pattern: using recursion for problems that are naturally iterative

# bad — recursive sum of a flat list
def recursive_sum(values):
    if not values:
        return 0
    return values[0] + recursive_sum(values[1:])

# good — use the built-in
print(sum([1, 2, 3, 4, 5]))   # 15 — simpler, faster, no stack limit

# recursion shines for problems with self-similar structure:
# trees, graphs, nested data, divide-and-conquer algorithms
# it is unnecessary for flat sequences

## Interview Signals

- What are the two required parts of every recursive function?
- What happens in memory each time a recursive function calls itself?
- Why does Python not support tail call optimisation and what is the consequence?
- When would you choose recursion over iteration?

## Exercise

In [ ]:
def deep_get(data, path):
    """
    Retrieves a value from a nested dict using a dot-separated path string.
    Returns None if any key in the path does not exist.

    Example:
        deep_get({"a": {"b": {"c": 42}}}, "a.b.c")  => 42
        deep_get({"a": {"b": 1}},          "a.x.c")  => None
        deep_get({"a": 1},                 "a")       => 1

    Must be implemented recursively.
    Bug: the function body is missing — implement it.
    """
    pass


config = {
    "database": {
        "host": "localhost",
        "port": 5432,
        "credentials": {
            "user": "admin",
            "password": "secret"
        }
    },
    "region": "us-east",
    "retries": 3
}

assert deep_get(config, "region")                          == "us-east"
assert deep_get(config, "retries")                         == 3
assert deep_get(config, "database.host")                   == "localhost"
assert deep_get(config, "database.port")                   == 5432
assert deep_get(config, "database.credentials.user")       == "admin"
assert deep_get(config, "database.credentials.password")   == "secret"
assert deep_get(config, "database.missing")                is None
assert deep_get(config, "missing.key")                     is None
assert deep_get(config, "database.credentials.missing")    is None

print("all assertions passed")